# Walk-Forward Momentum

Tests momentum signals across successive training and trading windows.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
ENTROPY-WEIGHTED MOMENTUM STRATEGY - CORRECTED VERSION
======================================================

CRITICAL FIXES APPLIED:
-----------------------
1. Cash-inclusive equity for sizing and returns
2. Benchmark excluded from trading universe
3. Proportional transaction costs (removed tight clamps)
4. Fixed entropy calculation (epsilon for all bins)
5. Adjusted prices for splits/dividends
6. Removed unused STOP_LOSS parameters
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy.stats import entropy as scipy_entropy
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import warnings

warnings.filterwarnings("ignore")


class StrategyConfig:
    """Centralized configuration - all parameters configurable."""

    END_DATE = "2018-12-31"
    START_DATE = "2013-01-01"
    MIN_TRAINING_DAYS = 252
    TESTING_WINDOW = 63
    STEP_SIZE = 63
    INITIAL_CAPITAL = 100000.0
    BENCHMARK_TICKER = "URTH"
    MOMENTUM_LOOKBACK = 63
    ENTROPY_WINDOW = 63
    HISTOGRAM_BINS = 10
    ENTROPY_EPSILON = 1e-10
    MAX_ENTROPY = np.log(10)
    MIN_ASSETS = 5
    MAX_ASSETS = 15
    MIN_WEIGHT = 0.05
    MAX_WEIGHT = 0.2
    TARGET_EXPOSURE = 1.0
    REBALANCE_FREQUENCY = 21
    MIN_WEIGHT_CHANGE = 0.03
    COMMISSION_RATE = 0.0005
    BID_ASK_SPREAD = 0.0003
    MARKET_IMPACT = 0.0002
    ASSET_UNIVERSE = [
        "SPY",
        "QQQ",
        "IWM",
        "DIA",
        "VTI",
        "XLK",
        "XLF",
        "XLV",
        "XLE",
        "XLI",
        "XLY",
        "XLP",
        "XLU",
        "XLRE",
        "XLB",
        "EFA",
        "EEM",
        "VWO",
        "EWJ",
        "EWG",
        "EWU",
        "EWC",
        "AAPL",
        "MSFT",
        "GOOGL",
        "AMZN",
        "META",
        "NVDA",
        "TSLA",
        "TLT",
        "IEF",
        "AGG",
        "LQD",
        "GLD",
        "SLV",
        "USO",
        "VNQ",
        "JNJ",
        "PG",
        "WMT",
        "JPM",
        "BAC",
        "V",
        "MA",
    ]


class DataFetcher:
    """Fetch data with adjusted prices for splits/dividends."""

    def fetch_historical_data(
        self, ticker: str, start_date: str, end_date: str
    ) -> Optional[pd.DataFrame]:
        """
        Fetch adjusted OHLCV data.

        FIX: Uses adjClose for accurate returns across splits/dividends.
        """
        buffer_start = (pd.to_datetime(start_date) - timedelta(days=150)).strftime("%Y-%m-%d")
        try:
            response_prices = {"historical": download_price_records(ticker, buffer_start, end_date)}
            data = response_prices
            if "historical" not in data or len(data["historical"]) < 100:
                return None
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df = df.rename(
                columns={
                    "date": "Date",
                    "adjClose": "AdjClose",
                    "open": "Open",
                    "close": "Close",
                    "volume": "Volume",
                }
            )
            if "AdjClose" in df.columns and df["AdjClose"].notna().any():
                df["Price"] = df["AdjClose"]
            else:
                df["Price"] = df["Close"]
            null_pct = df["Price"].isnull().sum() / len(df)
            if null_pct > 0.05:
                return None
            return df[["Date", "Open", "Close", "Price", "Volume"]]
        except Exception as e:
            return None

    def fetch_universe_data(
        self, tickers: List[str], start_date: str, end_date: str
    ) -> Dict[str, pd.DataFrame]:
        """Fetch data for trading universe."""
        print(f"\nFetching adjusted price data for {len(tickers)} assets...")
        data_dict = {}
        successful = 0
        for ticker in tickers:
            df = self.fetch_historical_data(ticker, start_date, end_date)
            if df is not None:
                data_dict[ticker] = df
                successful += 1
                if successful <= 5:
                    print(f"  ✓ {ticker}: {len(df)} days")
        print(f"\n✓ Successfully fetched {successful}/{len(tickers)} assets")
        if successful < 5:
            raise ValueError(f"Insufficient assets: {successful} < 5")
        return data_dict


class EntropyMomentumCalculator:
    """
    Calculate entropy-weighted momentum with corrected entropy formula.

    FIX: Add epsilon to ALL bins before normalization (not drop zeros).
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.momentum_lookback = config.MOMENTUM_LOOKBACK
        self.entropy_window = config.ENTROPY_WINDOW
        self.histogram_bins = config.HISTOGRAM_BINS
        self.entropy_epsilon = config.ENTROPY_EPSILON
        self.max_entropy = config.MAX_ENTROPY

    def calculate_entropy_from_returns(self, returns: pd.Series) -> float:
        """
        Calculate Shannon entropy with proper numerical stability.

        FIX: Add epsilon to ALL bins before normalization.
        This prevents log(0) and gives more stable entropy estimates.
        """
        clean_returns = returns.dropna()
        if len(clean_returns) < 10:
            return self.max_entropy
        counts, _ = np.histogram(clean_returns, bins=self.histogram_bins)
        counts_with_epsilon = counts + self.entropy_epsilon
        probabilities = counts_with_epsilon / counts_with_epsilon.sum()
        entropy_val = scipy_entropy(probabilities)
        entropy_val = min(entropy_val, self.max_entropy)
        return entropy_val

    def calculate_features_t_minus_1(self, price_data: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate features using adjusted prices with T-1 lag.

        Uses 'Price' column which contains adjusted close prices.
        """
        df = price_data.copy()
        df = df.sort_values("Date").reset_index(drop=True)
        df["Returns"] = df["Price"].pct_change()
        df["Momentum_Raw"] = df["Price"] / df["Price"].shift(self.momentum_lookback) - 1
        entropy_values = []
        for i in range(len(df)):
            if i < self.entropy_window:
                entropy_values.append(np.nan)
            else:
                hist_returns = df["Returns"].iloc[i - self.entropy_window : i]
                entropy_val = self.calculate_entropy_from_returns(hist_returns)
                entropy_values.append(entropy_val)
        df["Entropy_Raw"] = entropy_values
        df["Reliability_Raw"] = 1.0 / df["Entropy_Raw"]
        df["Signal_Raw"] = df["Momentum_Raw"] * df["Reliability_Raw"]
        df["Momentum"] = df["Momentum_Raw"].shift(1)
        df["Entropy"] = df["Entropy_Raw"].shift(1)
        df["Reliability"] = df["Reliability_Raw"].shift(1)
        df["Signal"] = df["Signal_Raw"].shift(1)
        df = df.drop(["Momentum_Raw", "Entropy_Raw", "Reliability_Raw", "Signal_Raw"], axis=1)
        return df

    def calculate_universe_signals(
        self, data_dict: Dict[str, pd.DataFrame], benchmark_ticker: str
    ) -> pd.DataFrame:
        """
        Calculate signals for universe.

        FIX: Explicitly exclude benchmark from signal matrix.
        """
        print("\nCalculating entropy-weighted momentum signals...")
        signal_dict = {}
        momentum_dict = {}
        entropy_dict = {}
        for ticker, price_df in data_dict.items():
            if ticker == benchmark_ticker:
                continue
            features_df = self.calculate_features_t_minus_1(price_df)
            features_df = features_df.set_index("Date")
            signal_dict[ticker] = features_df["Signal"]
            momentum_dict[ticker] = features_df["Momentum"]
            entropy_dict[ticker] = features_df["Entropy"]
        signals_df = pd.DataFrame(signal_dict)
        momentum_df = pd.DataFrame(momentum_dict)
        entropy_df = pd.DataFrame(entropy_dict)
        print(f"✓ Signals calculated for {len(signals_df.columns)} tradeable assets")
        print(f"  (Benchmark {benchmark_ticker} excluded from trading)")
        print(f"  Date range: {signals_df.index.min().date()} to {signals_df.index.max().date()}")
        signals_df.momentum = momentum_df
        signals_df.entropy = entropy_df
        return signals_df


class EntropyMomentumOptimizer:
    """Portfolio optimizer with position limits."""

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.min_assets = config.MIN_ASSETS
        self.max_assets = config.MAX_ASSETS
        self.min_weight = config.MIN_WEIGHT
        self.max_weight = config.MAX_WEIGHT
        self.target_exposure = config.TARGET_EXPOSURE

    def optimize_portfolio(
        self, signals: pd.Series, current_positions: Optional[Dict] = None
    ) -> Dict[str, float]:
        """
        Optimize portfolio weights from entropy-momentum signals.
        """
        positive_signals = signals[signals > 0].dropna()
        if len(positive_signals) == 0:
            return {}
        ranked_signals = positive_signals.sort_values(ascending=False)
        n_select = min(self.max_assets, len(ranked_signals))
        n_select = (
            max(n_select, self.min_assets)
            if len(ranked_signals) >= self.min_assets
            else len(ranked_signals)
        )
        top_signals = ranked_signals.head(n_select)
        raw_weights = top_signals / top_signals.sum()
        constrained_weights = raw_weights.clip(lower=self.min_weight, upper=self.max_weight)
        total_weight = constrained_weights.sum()
        if total_weight > 0:
            final_weights = constrained_weights / total_weight * self.target_exposure
        else:
            return {}
        if current_positions is not None:
            significant_change = False
            for ticker in set(list(final_weights.index) + list(current_positions.keys())):
                old_weight = current_positions.get(ticker, 0)
                new_weight = final_weights.get(ticker, 0)
                if abs(new_weight - old_weight) > self.config.MIN_WEIGHT_CHANGE:
                    significant_change = True
                    break
            if not significant_change:
                return current_positions
        return final_weights.to_dict()


class BacktestEngine:
    """
    Backtest engine with cash-inclusive equity tracking.

    FIX: Track cash separately, use total_equity for sizing and returns.
    FIX: Proportional transaction costs (no artificial clamps).
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.trades = []

    def calculate_transaction_costs(self, trade_value: float) -> float:
        """
        Proportional transaction costs.

        FIX: Removed tight $1-$5 clamp.
        Costs scale naturally with trade size.
        """
        abs_value = abs(trade_value)
        commission = abs_value * self.config.COMMISSION_RATE
        spread_cost = abs_value * self.config.BID_ASK_SPREAD
        impact_cost = abs_value * self.config.MARKET_IMPACT
        return commission + spread_cost + impact_cost

    def run_backtest(
        self,
        data_dict: Dict[str, pd.DataFrame],
        signals_df: pd.DataFrame,
        optimizer: EntropyMomentumOptimizer,
        benchmark_ticker: str,
    ) -> Dict:
        """
        Execute backtest with cash-inclusive equity.

        FIX: Portfolio value = risky_value + cash
        All sizing and returns based on total equity.
        """
        print("\n" + "=" * 80)
        print("EXECUTING BACKTEST (CASH-INCLUSIVE EQUITY)")
        print("=" * 80)
        start_date = pd.to_datetime(self.config.START_DATE)
        end_date = pd.to_datetime(self.config.END_DATE)
        backtest_dates = signals_df.index[
            (signals_df.index >= start_date) & (signals_df.index <= end_date)
        ]
        if len(backtest_dates) < 50:
            print(f"ERROR: Insufficient dates ({len(backtest_dates)})")
            return self._empty_results(signals_df, data_dict)
        print(f"Period: {backtest_dates[0].date()} to {backtest_dates[-1].date()}")
        print(f"Trading days: {len(backtest_dates)}")
        cash = self.config.INITIAL_CAPITAL
        positions = {}
        portfolio_history = []
        last_rebalance = -self.config.REBALANCE_FREQUENCY
        total_costs = 0.0
        for day_idx, signal_date in enumerate(backtest_dates[:-1]):
            current_prices = {}
            for ticker in signals_df.columns:
                if ticker in data_dict:
                    ticker_df = data_dict[ticker]
                    matches = ticker_df[ticker_df["Date"] == signal_date]
                    if not matches.empty:
                        current_prices[ticker] = matches.iloc[0]["Price"]
            risky_value = sum(
                (positions.get(ticker, 0) * current_prices.get(ticker, 0) for ticker in positions)
            )
            total_equity = cash + risky_value
            if day_idx == 0:
                total_equity = self.config.INITIAL_CAPITAL
            if day_idx % 50 == 0:
                print(
                    f"  Day {day_idx + 1}: {signal_date.date()} | Equity: ${total_equity:,.0f} (Cash: ${cash:,.0f})"
                )
            days_since_rebal = day_idx - last_rebalance
            should_rebalance = days_since_rebal >= self.config.REBALANCE_FREQUENCY
            if should_rebalance:
                current_signals = signals_df.loc[signal_date]
                current_weights = {}
                for ticker, shares in positions.items():
                    if ticker in current_prices and total_equity > 0:
                        current_weights[ticker] = shares * current_prices[ticker] / total_equity
                target_weights = optimizer.optimize_portfolio(current_signals, current_weights)
                exec_idx = day_idx + 1
                if exec_idx >= len(backtest_dates):
                    continue
                exec_date = backtest_dates[exec_idx]
                exec_prices = {}
                for ticker in signals_df.columns:
                    if ticker in data_dict:
                        ticker_df = data_dict[ticker]
                        matches = ticker_df[ticker_df["Date"] == exec_date]
                        if not matches.empty:
                            exec_prices[ticker] = matches.iloc[0]["Open"]
                day_costs = 0.0
                for ticker in list(positions.keys()):
                    if ticker not in target_weights and ticker in exec_prices:
                        shares = positions[ticker]
                        value = shares * exec_prices[ticker]
                        cost = self.calculate_transaction_costs(value)
                        cash += value - cost
                        day_costs += cost
                        del positions[ticker]
                        self.trades.append(
                            {
                                "date": exec_date,
                                "ticker": ticker,
                                "action": "SELL",
                                "shares": shares,
                                "price": exec_prices[ticker],
                                "value": value,
                                "cost": cost,
                            }
                        )
                for ticker, target_weight in target_weights.items():
                    if ticker not in exec_prices:
                        continue
                    target_value = total_equity * target_weight
                    current_value = positions.get(ticker, 0) * exec_prices[ticker]
                    trade_value = target_value - current_value
                    if abs(trade_value) > total_equity * 0.01:
                        shares_to_trade = trade_value / exec_prices[ticker]
                        cost = self.calculate_transaction_costs(abs(trade_value))
                        positions[ticker] = positions.get(ticker, 0) + shares_to_trade
                        cash -= trade_value + cost
                        day_costs += cost
                        self.trades.append(
                            {
                                "date": exec_date,
                                "ticker": ticker,
                                "action": "BUY" if shares_to_trade > 0 else "SELL",
                                "shares": abs(shares_to_trade),
                                "price": exec_prices[ticker],
                                "value": abs(trade_value),
                                "cost": cost,
                            }
                        )
                total_costs += day_costs
                last_rebalance = day_idx
            portfolio_history.append(
                {
                    "Date": signal_date,
                    "Total_Equity": total_equity,
                    "Cash": cash,
                    "Risky_Value": risky_value,
                    "Positions": len(positions),
                }
            )
        portfolio_df = pd.DataFrame(portfolio_history)
        portfolio_df["Returns"] = portfolio_df["Total_Equity"].pct_change()
        final_value = portfolio_df["Total_Equity"].iloc[-1]
        total_return = final_value / self.config.INITIAL_CAPITAL - 1
        days = len(portfolio_df)
        annual_return = (1 + total_return) ** (252 / days) - 1 if days > 0 else 0
        returns = portfolio_df["Returns"].dropna()
        volatility = returns.std() * np.sqrt(252) if len(returns) > 0 else 0
        sharpe = (annual_return - 0.02) / volatility if volatility > 0.001 else 0.0
        cumulative = (1 + returns).cumprod() if len(returns) > 0 else pd.Series([1])
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min() if len(drawdown) > 0 else 0
        results = {
            "portfolio_history": portfolio_df,
            "trade_history": self.trades,
            "metrics": {
                "initial_capital": self.config.INITIAL_CAPITAL,
                "final_value": final_value,
                "total_return": total_return,
                "annual_return": annual_return,
                "volatility": volatility,
                "sharpe_ratio": sharpe,
                "max_drawdown": max_dd,
                "total_costs": total_costs,
                "num_trades": len(self.trades),
            },
            "signals_df": signals_df,
            "data_dict": data_dict,
        }
        print("\n" + "=" * 80)
        print("BACKTEST COMPLETE")
        print("=" * 80)
        print(f"Final Equity: ${final_value:,.0f}")
        print(f"Total Return: {total_return:.2%}")
        print(f"Annual Return: {annual_return:.2%}")
        print(f"Sharpe Ratio: {sharpe:.2f}")
        print(f"Max Drawdown: {max_dd:.2%}")
        print(f"Transaction Costs: ${total_costs:,.0f}")
        print(f"Trades: {len(self.trades)}")
        print("=" * 80)
        return results

    def _empty_results(self, signals_df, data_dict):
        """Return empty results structure for error cases."""
        return {
            "portfolio_history": pd.DataFrame(),
            "trade_history": [],
            "metrics": {
                "initial_capital": self.config.INITIAL_CAPITAL,
                "final_value": self.config.INITIAL_CAPITAL,
                "total_return": 0.0,
                "annual_return": 0.0,
                "volatility": 0.0,
                "sharpe_ratio": 0.0,
                "max_drawdown": 0.0,
                "total_costs": 0.0,
                "num_trades": 0,
            },
            "signals_df": signals_df,
            "data_dict": data_dict,
        }


class WalkForwardValidator:
    """
    Walk-forward validation with expanding windows.

    METHODOLOGY:
    ------------
    1. Start with MIN_TRAINING_DAYS of data
    2. Test on next TESTING_WINDOW days (out-of-sample)
    3. Expand training window by STEP_SIZE
    4. Repeat until end of data
    5. Track performance by fold for stability analysis

    This ensures true out-of-sample testing while maintaining
    the T-1 lag protection in signal calculation.
    """

    def __init__(self, config: StrategyConfig):
        self.config = config

    def create_folds(self, all_dates: pd.DatetimeIndex) -> List[Dict]:
        """
        Create walk-forward folds with expanding windows.

        Returns list of fold definitions with train/test periods.
        """
        all_dates = sorted(all_dates)
        min_train = self.config.MIN_TRAINING_DAYS
        test_window = self.config.TESTING_WINDOW
        step_size = self.config.STEP_SIZE
        folds = []
        fold_num = 0
        current_pos = min_train
        while current_pos + test_window <= len(all_dates):
            fold_num += 1
            train_start_idx = 0
            train_end_idx = current_pos
            test_start_idx = current_pos
            test_end_idx = min(current_pos + test_window, len(all_dates))
            folds.append(
                {
                    "fold_num": fold_num,
                    "train_start_idx": train_start_idx,
                    "train_end_idx": train_end_idx,
                    "test_start_idx": test_start_idx,
                    "test_end_idx": test_end_idx,
                    "train_start_date": all_dates[train_start_idx],
                    "train_end_date": all_dates[train_end_idx - 1],
                    "test_start_date": all_dates[test_start_idx],
                    "test_end_date": all_dates[test_end_idx - 1],
                    "train_days": train_end_idx - train_start_idx,
                    "test_days": test_end_idx - test_start_idx,
                }
            )
            current_pos += step_size
        return folds

    def run_fold(
        self,
        fold: Dict,
        data_dict: Dict[str, pd.DataFrame],
        calculator: EntropyMomentumCalculator,
        optimizer: EntropyMomentumOptimizer,
        engine: BacktestEngine,
        benchmark_ticker: str,
    ) -> Dict:
        """
        Execute backtest for a single fold.

        Training data used for signal calculation (expanding window).
        Testing data for out-of-sample performance evaluation.
        """
        all_dates = []
        for df in data_dict.values():
            all_dates.extend(df["Date"].tolist())
        all_dates = sorted(set(all_dates))
        train_dates = all_dates[fold["train_start_idx"] : fold["train_end_idx"]]
        test_dates = all_dates[fold["test_start_idx"] : fold["test_end_idx"]]
        fold_data = {}
        for ticker, df in data_dict.items():
            buffer_start = train_dates[0] - timedelta(days=150)
            fold_end = test_dates[-1]
            fold_df = df[(df["Date"] >= buffer_start) & (df["Date"] <= fold_end)].copy()
            if len(fold_df) >= 100:
                fold_data[ticker] = fold_df
        if len(fold_data) < 5:
            return None
        signals_df = calculator.calculate_universe_signals(fold_data, benchmark_ticker)
        test_signals = signals_df[
            (signals_df.index >= test_dates[0]) & (signals_df.index <= test_dates[-1])
        ]
        if len(test_signals) < 10:
            return None
        fold_engine = BacktestEngine(self.config)
        original_start = self.config.START_DATE
        original_end = self.config.END_DATE
        self.config.START_DATE = test_dates[0].strftime("%Y-%m-%d")
        self.config.END_DATE = test_dates[-1].strftime("%Y-%m-%d")
        results = fold_engine.run_backtest(fold_data, signals_df, optimizer, benchmark_ticker)
        self.config.START_DATE = original_start
        self.config.END_DATE = original_end
        return results

    def validate(
        self,
        data_dict: Dict[str, pd.DataFrame],
        calculator: EntropyMomentumCalculator,
        optimizer: EntropyMomentumOptimizer,
        benchmark_ticker: str,
    ) -> Dict:
        """
        Execute full walk-forward validation.

        Returns comprehensive validation report with per-fold results.
        """
        print("\n" + "=" * 80)
        print("WALK-FORWARD VALIDATION (EXPANDING WINDOW)")
        print("=" * 80)
        all_dates = []
        for df in data_dict.values():
            all_dates.extend(df["Date"].tolist())
        all_dates = pd.DatetimeIndex(sorted(set(all_dates)))
        print(f"\nFull period: {all_dates[0].date()} to {all_dates[-1].date()}")
        print(f"Total days: {len(all_dates)}")
        print(f"\nWalk-forward configuration:")
        print(f"  Minimum training: {self.config.MIN_TRAINING_DAYS} days")
        print(f"  Testing window: {self.config.TESTING_WINDOW} days")
        print(f"  Step size: {self.config.STEP_SIZE} days")
        folds = self.create_folds(all_dates)
        print(f"\nTotal folds: {len(folds)}")
        fold_results = []
        engine = BacktestEngine(self.config)
        for fold in folds:
            print(f"\n{'=' * 80}")
            print(f"FOLD {fold['fold_num']}/{len(folds)}")
            print(f"{'=' * 80}")
            print(
                f"Training: {fold['train_start_date'].date()} to {fold['train_end_date'].date()} ({fold['train_days']} days)"
            )
            print(
                f"Testing:  {fold['test_start_date'].date()} to {fold['test_end_date'].date()} ({fold['test_days']} days)"
            )
            results = self.run_fold(
                fold, data_dict, calculator, optimizer, engine, benchmark_ticker
            )
            if results is None or results["portfolio_history"].empty:
                print(f"  ✗ Fold {fold['fold_num']}: Insufficient data")
                continue
            metrics = results["metrics"]
            fold_results.append(
                {
                    "fold_num": fold["fold_num"],
                    "train_start": fold["train_start_date"],
                    "train_end": fold["train_end_date"],
                    "test_start": fold["test_start_date"],
                    "test_end": fold["test_end_date"],
                    "total_return": metrics["total_return"],
                    "annual_return": metrics["annual_return"],
                    "volatility": metrics["volatility"],
                    "sharpe_ratio": metrics["sharpe_ratio"],
                    "max_drawdown": metrics["max_drawdown"],
                    "num_trades": metrics["num_trades"],
                    "total_costs": metrics["total_costs"],
                }
            )
            print(
                f"  ✓ Return: {metrics['total_return']:>7.2%} | Sharpe: {metrics['sharpe_ratio']:>5.2f} | MaxDD: {metrics['max_drawdown']:>6.2%} | Trades: {metrics['num_trades']}"
            )
        if not fold_results:
            print("\n✗ No successful folds")
            return {"folds": [], "summary": {}}
        df_folds = pd.DataFrame(fold_results)
        print("\n" + "=" * 80)
        print("WALK-FORWARD SUMMARY")
        print("=" * 80)
        summary = {
            "num_folds": len(fold_results),
            "mean_return": df_folds["annual_return"].mean(),
            "std_return": df_folds["annual_return"].std(),
            "mean_sharpe": df_folds["sharpe_ratio"].mean(),
            "std_sharpe": df_folds["sharpe_ratio"].std(),
            "mean_drawdown": df_folds["max_drawdown"].mean(),
            "worst_drawdown": df_folds["max_drawdown"].min(),
            "positive_folds": (df_folds["total_return"] > 0).sum(),
            "negative_folds": (df_folds["total_return"] <= 0).sum(),
            "win_rate": (df_folds["total_return"] > 0).mean(),
            "total_trades": df_folds["num_trades"].sum(),
        }
        print(f"\nPerformance across {summary['num_folds']} folds:")
        print(
            f"  Mean Annual Return: {summary['mean_return']:>7.2%} ± {summary['std_return']:>6.2%}"
        )
        print(
            f"  Mean Sharpe Ratio:  {summary['mean_sharpe']:>7.2f} ± {summary['std_sharpe']:>6.2f}"
        )
        print(f"  Mean Max Drawdown:  {summary['mean_drawdown']:>7.2%}")
        print(f"  Worst Drawdown:     {summary['worst_drawdown']:>7.2%}")
        print(
            f"  Win Rate:           {summary['win_rate']:>7.1%} ({summary['positive_folds']}/{summary['num_folds']})"
        )
        print(f"  Total Trades:       {summary['total_trades']}")
        sharpe_cv = (
            abs(summary["std_sharpe"] / summary["mean_sharpe"])
            if summary["mean_sharpe"] != 0
            else np.inf
        )
        return_cv = (
            abs(summary["std_return"] / summary["mean_return"])
            if summary["mean_return"] != 0
            else np.inf
        )
        print(f"\nStability Metrics:")
        print(f"  Sharpe CV: {sharpe_cv:.2f} {('✓ STABLE' if sharpe_cv < 0.5 else '✗ UNSTABLE')}")
        print(f"  Return CV: {return_cv:.2f} {('✓ STABLE' if return_cv < 0.5 else '✗ UNSTABLE')}")
        print("=" * 80)
        return {"folds": fold_results, "summary": summary, "fold_dataframe": df_folds}


def create_visualizations(results: Dict, config: StrategyConfig):
    """Create comprehensive dashboard."""
    portfolio_df = results["portfolio_history"]
    if portfolio_df.empty:
        print("No data to visualize")
        return
    signals_df = results["signals_df"]
    data_dict = results["data_dict"]
    benchmark_data = None
    if config.BENCHMARK_TICKER in data_dict:
        bench_df = data_dict[config.BENCHMARK_TICKER].copy()
        bench_df = bench_df[bench_df["Date"].isin(portfolio_df["Date"])]
        if not bench_df.empty:
            first_price = bench_df["Price"].iloc[0]
            bench_df["Value"] = bench_df["Price"] / first_price * config.INITIAL_CAPITAL
            benchmark_data = bench_df.set_index("Date")["Value"]
    fig = plt.figure(figsize=(20, 14))
    gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.25)
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(
        portfolio_df["Date"],
        portfolio_df["Total_Equity"],
        label="Strategy",
        linewidth=2.5,
        color="#2E86AB",
    )
    if benchmark_data is not None:
        ax1.plot(
            benchmark_data.index,
            benchmark_data.values,
            label=f"{config.BENCHMARK_TICKER}",
            linewidth=2,
            color="#A23B72",
            alpha=0.7,
        )
    ax1.set_title("Entropy-Weighted Momentum Strategy", fontsize=16, fontweight="bold")
    ax1.set_ylabel("Portfolio Value ($)", fontsize=12)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.fill_between(
        portfolio_df["Date"], 0, portfolio_df["Cash"], alpha=0.5, color="green", label="Cash"
    )
    ax2.fill_between(
        portfolio_df["Date"],
        portfolio_df["Cash"],
        portfolio_df["Total_Equity"],
        alpha=0.5,
        color="blue",
        label="Risky Assets",
    )
    ax2.set_title("Portfolio Composition", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Value ($)", fontsize=11)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax3 = fig.add_subplot(gs[1, 1])
    returns = portfolio_df["Returns"].dropna()
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    ax3.fill_between(range(len(drawdown)), drawdown * 100, 0, alpha=0.7, color="red")
    ax3.set_title("Drawdown", fontsize=14, fontweight="bold")
    ax3.set_ylabel("Drawdown (%)", fontsize=11)
    ax3.grid(True, alpha=0.3)
    ax4 = fig.add_subplot(gs[2, 0])
    ax4.plot(portfolio_df["Date"], portfolio_df["Positions"], linewidth=2, color="blue")
    ax4.set_title("Active Positions", fontsize=14, fontweight="bold")
    ax4.set_ylabel("Count", fontsize=11)
    ax4.grid(True, alpha=0.3)
    ax5 = fig.add_subplot(gs[2, 1])
    all_signals = signals_df.values.flatten()
    all_signals = all_signals[~np.isnan(all_signals)]
    ax5.hist(all_signals, bins=50, alpha=0.7, color="purple", edgecolor="black")
    ax5.axvline(x=0, color="red", linestyle="--", linewidth=2)
    ax5.set_title("Signal Distribution", fontsize=14, fontweight="bold")
    ax5.set_xlabel("Entropy-Weighted Signal", fontsize=11)
    ax5.grid(True, alpha=0.3)
    ax6 = fig.add_subplot(gs[3, :])
    portfolio_df["Year"] = pd.to_datetime(portfolio_df["Date"]).dt.year
    portfolio_df["Month"] = pd.to_datetime(portfolio_df["Date"]).dt.month
    monthly_rets = (
        portfolio_df.groupby(["Year", "Month"])
        .apply(lambda x: (1 + x["Returns"]).prod() - 1)
        .reset_index(name="Return")
    )
    heatmap_data = monthly_rets.pivot(index="Year", columns="Month", values="Return")
    month_names = [
        "Jan",
        "Feb",
        "Mar",
        "Apr",
        "May",
        "Jun",
        "Jul",
        "Aug",
        "Sep",
        "Oct",
        "Nov",
        "Dec",
    ]
    heatmap_data.columns = [month_names[int(m) - 1] for m in heatmap_data.columns]
    sns.heatmap(
        heatmap_data * 100,
        annot=True,
        fmt=".1f",
        cmap="RdYlGn",
        center=0,
        ax=ax6,
        cbar_kws={"label": "Monthly Return (%)"},
    )
    ax6.set_title("Monthly Returns", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


def create_fold_performance_chart(validation_results: Dict):
    """
    Visualize walk-forward fold performance.

    Shows consistency and stability across out-of-sample periods.
    """
    if not validation_results["folds"]:
        return
    df_folds = validation_results["fold_dataframe"]
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    ax1 = axes[0, 0]
    colors = ["green" if r > 0 else "red" for r in df_folds["total_return"]]
    ax1.bar(df_folds["fold_num"], df_folds["total_return"] * 100, color=colors, alpha=0.7)
    ax1.axhline(y=0, color="black", linestyle="-", linewidth=1)
    ax1.set_title("Returns by Fold (Out-of-Sample)", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Fold Number")
    ax1.set_ylabel("Return (%)")
    ax1.grid(True, alpha=0.3)
    ax2 = axes[0, 1]
    ax2.plot(
        df_folds["fold_num"],
        df_folds["sharpe_ratio"],
        marker="o",
        linewidth=2,
        markersize=8,
        color="blue",
    )
    ax2.axhline(y=0, color="black", linestyle="-", linewidth=1)
    ax2.axhline(y=1, color="green", linestyle="--", linewidth=1, alpha=0.5)
    ax2.set_title("Sharpe Ratio by Fold", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Fold Number")
    ax2.set_ylabel("Sharpe Ratio")
    ax2.grid(True, alpha=0.3)
    ax3 = axes[1, 0]
    ax3.bar(df_folds["fold_num"], df_folds["max_drawdown"] * 100, color="red", alpha=0.7)
    ax3.set_title("Max Drawdown by Fold", fontsize=14, fontweight="bold")
    ax3.set_xlabel("Fold Number")
    ax3.set_ylabel("Drawdown (%)")
    ax3.grid(True, alpha=0.3)
    ax4 = axes[1, 1]
    ax4.axis("off")
    summary = validation_results["summary"]
    summary_text = f"\n    WALK-FORWARD VALIDATION SUMMARY\n    {'=' * 45}\n    \n    Number of Folds: {summary['num_folds']}\n    \n    Annual Return:\n      Mean: {summary['mean_return']:>7.2%}\n      Std:  {summary['std_return']:>7.2%}\n    \n    Sharpe Ratio:\n      Mean: {summary['mean_sharpe']:>7.2f}\n      Std:  {summary['std_sharpe']:>7.2f}\n    \n    Max Drawdown:\n      Mean:  {summary['mean_drawdown']:>7.2%}\n      Worst: {summary['worst_drawdown']:>7.2%}\n    \n    Win Rate: {summary['win_rate']:>7.1%}\n    Positive Folds: {summary['positive_folds']}/{summary['num_folds']}\n    \n    Total Trades: {summary['total_trades']}\n    "
    ax4.text(
        0.1,
        0.95,
        summary_text,
        transform=ax4.transAxes,
        fontsize=11,
        verticalalignment="top",
        fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.3),
    )
    plt.tight_layout()
    plt.show()


def main():
    """Execute entropy-weighted momentum with walk-forward validation."""
    print("=" * 80)
    print("ENTROPY-WEIGHTED MOMENTUM - WALK-FORWARD VALIDATION")
    print("=" * 80)
    print("\nFixes Applied:")
    print("  ✓ Cash-inclusive equity")
    print("  ✓ Benchmark excluded from trading")
    print("  ✓ Proportional transaction costs")
    print("  ✓ Fixed entropy calculation")
    print("  ✓ Adjusted prices")
    print("  ✓ Walk-forward expanding window validation")
    print("=" * 80)
    config = StrategyConfig()
    print(f"\nConfiguration:")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Benchmark: {config.BENCHMARK_TICKER} (comparison only)")
    print(f"  Universe: {len(config.ASSET_UNIVERSE)} assets")
    print(f"\nWalk-Forward Setup:")
    print(f"  Min Training: {config.MIN_TRAINING_DAYS} days")
    print(f"  Test Window: {config.TESTING_WINDOW} days")
    print(f"  Step Size: {config.STEP_SIZE} days")
    print("\n" + "=" * 80)
    print("STEP 1: DATA ACQUISITION")
    print("=" * 80)
    fetcher = DataFetcher()
    all_tickers = config.ASSET_UNIVERSE + [config.BENCHMARK_TICKER]
    data_dict = fetcher.fetch_universe_data(all_tickers, config.START_DATE, config.END_DATE)
    calculator = EntropyMomentumCalculator(config)
    optimizer = EntropyMomentumOptimizer(config)
    print("\n" + "=" * 80)
    print("STEP 2: WALK-FORWARD VALIDATION")
    print("=" * 80)
    validator = WalkForwardValidator(config)
    validation_results = validator.validate(
        data_dict, calculator, optimizer, config.BENCHMARK_TICKER
    )
    print("\n" + "=" * 80)
    print("STEP 3: FULL PERIOD BACKTEST (FOR VISUALIZATION)")
    print("=" * 80)
    signals_df = calculator.calculate_universe_signals(data_dict, config.BENCHMARK_TICKER)
    engine = BacktestEngine(config)
    full_results = engine.run_backtest(data_dict, signals_df, optimizer, config.BENCHMARK_TICKER)
    print("\n" + "=" * 80)
    print("STEP 4: VISUALIZATION")
    print("=" * 80)
    create_visualizations(full_results, config)
    if validation_results["folds"]:
        create_fold_performance_chart(validation_results)
    return (full_results, validation_results)


if __name__ == "__main__":
    full_results, validation_results = main()
